In [5]:
!pip install -q mlflow==3.16.0 hi-ml-multimodal

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 77.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 67.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 47.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.7/148.7 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 kB 11.8 MB/s eta 0:00:00


In [6]:
print("test")

test


In [7]:
import mlflow
import health_multimodal
print("mlflow", mlflow.__version__)
print("health_multimodal OK")

mlflow 3.16.0
health_multimodal OK


In [8]:
%%writefile inference.py
"""
Cœur d'inférence de l'Auditeur de Cohérence Médicale.

Reçoit une radiographie thoracique et son compte rendu, et renvoie
un verdict (cohérent ou incohérent) accompagné d'un score.

Inférence autonome : chargement de BioViL-T (public, licence MIT),
reconstruction du classifieur entraîné et chargement de ses poids
portables (model.safetensors), puis application du seuil de décision
de 0,5. Le compte rendu est tronqué aux 256 premiers jetons, exactement
comme à l'entraînement. Les modèles sont chargés sur le processeur, puis
basculés sur le GPU au moment de la prédiction lorsqu'un GPU est disponible.
"""

import os
import torch
import torch.nn as nn
from PIL import Image
from transformers import AutoTokenizer, AutoModel
from safetensors.torch import load_file

# Encodeur d'image de BioViL-T, fourni par le paquet health_multimodal
# (installation : hi-ml-multimodal).
from health_multimodal.image.model.pretrained import get_biovil_t_image_encoder
from health_multimodal.image.data.transforms import create_chest_xray_transform_for_inference


# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
MODEL_ID = "microsoft/BiomedVLP-BioViL-T"   # BioViL-T, public, licence MIT
WEIGHTS_PATH = "model.safetensors"           # poids portables du classifieur
SEUIL = 0.5                                  # >= 0,5 : cohérent ; sinon : incohérent
LONGUEUR_TEXTE = 256                          # nombre de jetons utilisés, comme à l'entraînement


def peripherique():
    """Renvoie le GPU si disponible au moment de l'appel, sinon le processeur."""
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ---------------------------------------------------------------------------
# Architecture du classifieur (identique au notebook de modélisation)
# ---------------------------------------------------------------------------
class VisualProjectionLayer(nn.Module):
    """Projette les patches d'image (dimension 128) vers la dimension du texte (768)."""

    def __init__(self, img_dim=128, text_dim=768):
        super().__init__()
        self.projector = nn.Linear(img_dim, text_dim)

    def forward(self, img_patches):
        x = img_patches.permute(0, 2, 3, 1)   # [batch, 14, 14, 128]
        x = x.flatten(1, 2)                    # [batch, 196, 128]
        return self.projector(x)               # [batch, 196, 768]


class CrossAttentionClassifierBiovil(nn.Module):
    """Bloc d'attention croisée : le texte interroge l'image, puis classification."""

    def __init__(self, embed_dim=768, num_heads=8, dropout=0.3):
        super().__init__()
        self.projection_layer = VisualProjectionLayer()
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=embed_dim, num_heads=num_heads, dropout=dropout, batch_first=True
        )
        self.layer_norm1 = nn.LayerNorm(embed_dim)
        self.layer_norm2 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 2, embed_dim),
        )
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.GELU(),
            nn.LayerNorm(256),
            nn.Dropout(dropout),
            nn.Linear(256, 1),
        )

    def forward(self, img_patches, text_tokens):
        img_patches_proj = self.projection_layer(img_patches)
        norm_text = self.layer_norm1(text_tokens)
        attn_output, _ = self.cross_attention(
            query=norm_text, key=img_patches_proj, value=img_patches_proj
        )
        x = attn_output + text_tokens
        x = self.layer_norm2(self.ffn(x)) + x
        x_pooled, _ = torch.max(x, dim=1)
        return self.classifier(x_pooled)       # score brut (logit), forme [batch, 1]


# ---------------------------------------------------------------------------
# Chargement (une seule fois) de BioViL-T et du classifieur, sur le processeur
# ---------------------------------------------------------------------------
_ressources = {}

def charger_modeles():
    if _ressources:
        return _ressources

    # Côté texte : tokenizer et modèle CXR-BERT de BioViL-T
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    text_model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True).eval()

    # Côté image : encodeur image de BioViL-T
    image_model = get_biovil_t_image_encoder().eval()
    image_transform = create_chest_xray_transform_for_inference(resize=512, center_crop_size=448)

    # Classifieur : architecture reconstruite. Les poids ne sont chargés que si le
    # fichier existe. En inférence, model.safetensors est présent, donc le classifieur
    # entraîné est chargé normalement. À l'entraînement (où ces poids sont justement
    # ce que l'on produit), le fichier peut être absent : on garde alors le classifieur
    # tel quel, sans erreur, car l'entraînement crée et entraîne son propre classifieur.
    classifieur = CrossAttentionClassifierBiovil()
    if os.path.exists(WEIGHTS_PATH):
        classifieur.load_state_dict(load_file(WEIGHTS_PATH))
    classifieur = classifieur.eval()

    _ressources.update(
        tokenizer=tokenizer,
        text_model=text_model,
        image_model=image_model,
        image_transform=image_transform,
        classifieur=classifieur,
    )
    return _ressources


# ---------------------------------------------------------------------------
# Préparation des entrées
# ---------------------------------------------------------------------------
def encoder_texte(compte_rendu, r, dev):
    """Transforme le compte rendu en une séquence de vecteurs, forme [1, 512, 768]."""
    inputs = r["tokenizer"](
        compte_rendu,
        padding="max_length",
        truncation=True,
        max_length=512,
        return_tensors="pt",
    ).to(dev)
    with torch.no_grad():
        sortie = r["text_model"](
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            return_dict=True,
        )
    return sortie.last_hidden_state


def encoder_image(image, r, dev):
    """Transforme la radiographie en patches, forme [1, 128, 14, 14]."""
    if not isinstance(image, Image.Image):
        image = Image.open(image)
    image = image.convert("L")
    tenseur = r["image_transform"](image).unsqueeze(0).to(dev)
    with torch.no_grad():
        sortie = r["image_model"](tenseur)
    return sortie.projected_patch_embeddings


# ---------------------------------------------------------------------------
# Prédiction complète : image + texte -> verdict
# ---------------------------------------------------------------------------
def predire(image, compte_rendu):
    """
    Renvoie un dictionnaire :
      - verdict     : "COHÉRENT" ou "INCOHÉRENT"
      - prediction  : 1 (cohérent) ou 0 (incohérent)
      - probabilite : P(cohérent), entre 0 et 1
      - confiance   : proximité au verdict rendu, entre 0,5 et 1
    """
    r = charger_modeles()
    dev = peripherique()

    # Bascule les modèles sur le périphérique courant (sans effet s'ils y sont déjà)
    r["text_model"].to(dev)
    r["image_model"].to(dev)
    r["classifieur"].to(dev)

    patches_image = encoder_image(image, r, dev)
    sequence_texte = encoder_texte(compte_rendu, r, dev)
    sequence_texte = sequence_texte[:, :LONGUEUR_TEXTE, :]   # 256 jetons, comme à l'entraînement

    with torch.no_grad():
        logit = r["classifieur"](patches_image, sequence_texte)
        probabilite = torch.sigmoid(logit).item()

    prediction = 1 if probabilite >= SEUIL else 0
    verdict = "COHÉRENT" if prediction == 1 else "INCOHÉRENT"
    confiance = probabilite if prediction == 1 else 1.0 - probabilite

    return {
        "verdict": verdict,
        "prediction": prediction,
        "probabilite": round(probabilite, 4),
        "confiance": round(confiance, 4),
    }


# ---------------------------------------------------------------------------
# Test manuel
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    exemple_image = "exemple_radio.png"
    exemple_texte = "No acute cardiopulmonary process."
    print(predire(exemple_image, exemple_texte))


Overwriting inference.py


In [9]:
%%writefile train.py
"""
Script d'entraînement (et de réentraînement) de l'Auditeur de Cohérence Médicale.

Étapes :
  1. Charge le jeu déjà étiqueté (une ligne par paire, colonne target : 1 = cohérent, 0 = incohérent).
  2. Extrait les caractéristiques BioViL-T et les enregistre sur le disque, une par une.
  3. Entraîne le classifieur à attention croisée (perte BCE, AdamW, arrêt anticipé).
  4. Enregistre l'expérience dans MLflow.
  5. Sauvegarde les meilleurs poids au format portable model.safetensors.

Source des données : un fichier unique déjà assemblé et étiqueté
(chexpert_sample_dataset.csv), avec la colonne target prête et la colonne report
(compte rendu complet). C'est la même recette que le notebook de modélisation
d'origine, qui lisait lui aussi un seul fichier déjà étiqueté.

Environnement : le script fonctionne à deux endroits sans modification. Il détecte
s'il tourne sur Kaggle (où les données sont montées sous /kaggle/input) ou en local
(dossier CheXpert du projet), et choisit les chemins correspondants.

Mémoire : les caractéristiques extraites sont lues depuis le disque par petits lots,
au moment de l'entraînement, et jamais toutes chargées en mémoire d'un coup. Cela
permet d'entraîner sur l'ensemble des paires sans saturer la mémoire.

Robustesse et reprise : l'extraction sauvegarde chaque résultat sur le disque au
fur et à mesure. Si le programme s'interrompt, il suffit de le relancer : il repart
où il s'était arrêté, sans refaire ce qui est déjà en cache.
"""

import os
import gc
import hashlib
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from safetensors.torch import save_file
from tqdm import tqdm
import mlflow
from PIL import ImageFile

# Tolère les images tronquées, au lieu de faire planter le programme.
ImageFile.LOAD_TRUNCATED_IMAGES = True

from inference import (
    charger_modeles,
    encoder_image,
    encoder_texte,
    CrossAttentionClassifierBiovil,
    peripherique,
    LONGUEUR_TEXTE,
)


# ---------------------------------------------------------------------------
# Emplacement des données : détecté automatiquement selon l'environnement
# ---------------------------------------------------------------------------
# Le dossier /kaggle/input n'existe que sur Kaggle. Sa présence sert donc de
# repère : si je le trouve, je suis sur Kaggle ; sinon, je suis en local.
SUR_KAGGLE = os.path.exists("/kaggle/input")

if SUR_KAGGLE:
    # Sur Kaggle, mon jeu de données est monté en lecture seule sous ce chemin.
    # Le chemin inclut datasets/<identifiant>/<nom-du-dataset>, et mes images se
    # trouvent sous un double dossier images/images (issu de la compression du dossier).
    BASE_DONNEES = "/kaggle/input/datasets/aichajedha/auditeur-chexpert-20000"
    IMG_BASE = os.path.join(BASE_DONNEES, "images", "images")
else:
    # En local, les données sont dans le dossier CheXpert du projet.
    BASE_DONNEES = "../CheXpert"
    IMG_BASE = os.path.join(BASE_DONNEES, "CheXpert-v1.0-small")

# Le fichier de paires déjà étiqueté, au bon endroit dans les deux cas.
CSV_PAIRES = os.path.join(BASE_DONNEES, "chexpert_sample_dataset.csv")


# ---------------------------------------------------------------------------
# Réglages d'entraînement (mêmes valeurs que le notebook de modélisation)
# ---------------------------------------------------------------------------
N_SAMPLES = None         # None = tout le jeu (les 20 000 paires). Mettre un petit nombre (ex. 400) pour un test rapide.
EPOCHS = 100             # Nombre maximal d'époques, comme dans mon notebook (l'arrêt anticipé coupe souvent avant).
BATCH_SIZE = 64          # Taille de lot du classifieur, identique à mon notebook.
LR = 1e-5               # Taux d'apprentissage
WEIGHT_DECAY = 1e-2     # Régularisation L2
PATIENCE = 3            # Nombre d'époques sans amélioration avant l'arrêt anticipé
SEUIL = 0.5            # Seuil de décision cohérent / incohérent
NUM_WORKERS = 2         # Nombre de processus qui lisent le cache disque en parallèle
SORTIE_POIDS = "model.safetensors"
EXPERIENCE = "auditeur-coherence-medicale"

CACHE = "features_demo"                          # dossier des caractéristiques déjà extraites
FICHIER_EXCLUSIONS = "_images_a_ignorer.txt"     # images à sauter (corrompues)
FICHIER_EN_COURS = "_image_en_cours.txt"         # image en cours (témoin de plantage)


def cle(chemin, texte):
    """Identifiant stable d'une paire, pour nommer son fichier de cache."""
    # Une empreinte md5 du couple (chemin, texte) donne un nom de fichier unique
    # et reproductible : la même paire retombe toujours sur le même cache.
    return hashlib.md5((chemin + "||" + texte).encode("utf-8")).hexdigest()


def charger_exclusions():
    """Charge la liste des images à ignorer ; si un plantage a laissé un témoin, l'ajoute."""
    exclus = set()
    if os.path.exists(FICHIER_EXCLUSIONS):
        with open(FICHIER_EXCLUSIONS, encoding="utf-8") as f:
            exclus = {ligne.strip() for ligne in f if ligne.strip()}
    if os.path.exists(FICHIER_EN_COURS):
        with open(FICHIER_EN_COURS, encoding="utf-8") as f:
            coupable = f.read().strip()
        if coupable and coupable not in exclus:
            exclus.add(coupable)
            with open(FICHIER_EXCLUSIONS, "a", encoding="utf-8") as f:
                f.write(coupable + "\n")
            print(f"Image écartée (plantage précédent) :\n  {coupable}")
        os.remove(FICHIER_EN_COURS)
    return exclus


def construire_jeu(exclus):
    """Charge le jeu déjà étiqueté (colonne target) et compose les chemins d'images."""
    df = pd.read_csv(CSV_PAIRES)
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)

    lignes = []
    for _, r in df.iterrows():
        rel = str(r["path_to_image"])   # chemin relatif de l'image
        rep = str(r["report"])          # compte rendu complet (colonne report, comme à l'entraînement d'origine)

        # Garde-fou : j'écarte les comptes rendus vides ou quasi vides (peu exploitables).
        if len(rep.strip()) < 20:
            continue

        chemin = os.path.join(IMG_BASE, rel)

        # J'ignore les images exclues (corrompues) ou absentes du disque.
        if chemin in exclus or not os.path.exists(chemin):
            continue

        lignes.append((chemin, rep, int(r["target"])))

        # Si une limite est fixée (N_SAMPLES non None), on s'arrête une fois atteinte.
        if N_SAMPLES is not None and len(lignes) >= N_SAMPLES:
            break
    return lignes


def extraire_vers_cache(lignes, res, dev):
    """Extrait et sauvegarde sur disque les caractéristiques encore manquantes."""
    os.makedirs(CACHE, exist_ok=True)
    nouveaux = 0
    for chemin, texte, y in tqdm(lignes, desc="Extraction des caractéristiques"):
        dest = os.path.join(CACHE, cle(chemin, texte) + ".pt")
        if os.path.exists(dest):
            continue  # déjà extrait lors d'un lancement précédent

        # On note l'image en cours : en cas de plantage, elle sera écartée au prochain lancement.
        with open(FICHIER_EN_COURS, "w", encoding="utf-8") as f:
            f.write(chemin)

        # Encodage BioViL-T : patches de l'image et séquence de jetons du texte.
        patches = encoder_image(chemin, res, dev).squeeze(0).detach().cpu()
        seq = encoder_texte(texte, res, dev)[:, :LONGUEUR_TEXTE, :].squeeze(0).detach().cpu()
        torch.save({"img": patches, "txt": seq, "y": torch.tensor(float(y))}, dest)

        del patches, seq
        nouveaux += 1
        if nouveaux % 20 == 0:
            gc.collect()

    if os.path.exists(FICHIER_EN_COURS):
        os.remove(FICHIER_EN_COURS)
    print(f"{nouveaux} nouvelles paires extraites ; les autres étaient déjà en cache.")


class CacheDataset(Dataset):
    """Lit les caractéristiques d'une paire depuis le cache disque, à la demande.

    Ne garde en mémoire que la liste des références (chemin, texte, cible), pas les
    données elles-mêmes. Chaque lot est lu du disque au moment où l'entraînement en a
    besoin, puis libéré. La mémoire ne contient donc jamais qu'un lot à la fois.
    """

    def __init__(self, lignes):
        self.lignes = lignes

    def __len__(self):
        return len(self.lignes)

    def __getitem__(self, idx):
        chemin, texte, y = self.lignes[idx]
        dest = os.path.join(CACHE, cle(chemin, texte) + ".pt")
        d = torch.load(dest, weights_only=True)
        img = d["img"].float()
        txt = d["txt"].float()
        cible = torch.tensor(float(y))
        return img, txt, cible


def entrainer():
    dev = peripherique()
    print("Périphérique :", dev)

    # Chargement des encodeurs BioViL-T, nécessaires uniquement à l'extraction.
    res = charger_modeles()
    res["text_model"].to(dev)
    res["image_model"].to(dev)

    # Préparation du jeu : exclusions, puis sélection des paires exploitables.
    exclus = charger_exclusions()
    lignes = construire_jeu(exclus)
    print(f"{len(lignes)} paires sélectionnées")

    # Extraction (avec cache). Si tout est déjà en cache, cette étape est quasi instantanée.
    extraire_vers_cache(lignes, res, dev)

    # Les encodeurs ne servent plus après l'extraction : on rend la mémoire.
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Découpe entraînement / validation sur les indices, stratifiée pour garder
    # l'équilibre des classes. On ne manipule que des références, pas les données.
    indices = list(range(len(lignes)))
    cibles = [y for (_, _, y) in lignes]
    idx_tr, idx_va = train_test_split(
        indices, test_size=0.2, random_state=42, stratify=cibles
    )
    lignes_tr = [lignes[i] for i in idx_tr]
    lignes_va = [lignes[i] for i in idx_va]

    # Chargeurs par lots : ils lisent le cache disque au fur et à mesure (mémoire basse).
    train_loader = DataLoader(
        CacheDataset(lignes_tr), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS
    )
    val_loader = DataLoader(
        CacheDataset(lignes_va), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
    )

    # Classifieur à attention croisée : c'est la partie entraînée (BioViL-T reste gelé).
    model = CrossAttentionClassifierBiovil().to(dev)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=2
    )

    # Adresse du serveur MLflow où envoyer les résultats.
    # Lue depuis la variable d'environnement MLFLOW_TRACKING_URI (secret Kaggle) si elle existe ;
    # sinon, on utilise par défaut l'adresse de mon serveur MLflow hébergé sur Azure.
    adresse_mlflow = os.environ.get(
        "MLFLOW_TRACKING_URI",
        "https://auditeur-mlflow-hbb4ambmcnhhfjcz.swedencentral-01.azurewebsites.net",
    )
    mlflow.set_tracking_uri(adresse_mlflow)
    print("Serveur MLflow :", adresse_mlflow)

    mlflow.set_experiment(EXPERIENCE)
    with mlflow.start_run():
        mlflow.log_params({
            "n_samples": len(lignes),
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "learning_rate": LR,
            "weight_decay": WEIGHT_DECAY,
            "patience": PATIENCE,
            "text_length": LONGUEUR_TEXTE,
        })

        best_val = float("inf")
        best_state = None
        patience_ct = 0

        for epoch in range(EPOCHS):
            # --- Phase d'entraînement ---
            model.train()
            total, correct = 0.0, 0
            for xi, xt, yb in train_loader:
                xi, xt, yb = xi.to(dev), xt.to(dev), yb.to(dev)
                optimizer.zero_grad()
                out = model(xi, xt).squeeze(1)
                loss = criterion(out, yb)
                loss.backward()
                optimizer.step()
                total += loss.item()
                correct += ((torch.sigmoid(out) >= SEUIL).float() == yb).sum().item()
            tr_loss = total / len(train_loader)
            tr_acc = correct / len(train_loader.dataset)

            # --- Phase de validation (sans mise à jour des poids) ---
            model.eval()
            v_total, v_correct = 0.0, 0
            with torch.no_grad():
                for xi, xt, yb in val_loader:
                    xi, xt, yb = xi.to(dev), xt.to(dev), yb.to(dev)
                    out = model(xi, xt).squeeze(1)
                    v_total += criterion(out, yb).item()
                    v_correct += ((torch.sigmoid(out) >= SEUIL).float() == yb).sum().item()
            v_loss = v_total / len(val_loader)
            v_acc = v_correct / len(val_loader.dataset)
            scheduler.step(v_loss)

            mlflow.log_metric("train_loss", tr_loss, step=epoch)
            mlflow.log_metric("val_loss", v_loss, step=epoch)
            mlflow.log_metric("train_accuracy", tr_acc, step=epoch)
            mlflow.log_metric("val_accuracy", v_acc, step=epoch)
            print(
                f"Epoch {epoch + 1}/{EPOCHS}  "
                f"train_loss={tr_loss:.4f}  val_loss={v_loss:.4f}  val_acc={v_acc:.4f}"
            )

            if v_loss < best_val:
                best_val = v_loss
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                patience_ct = 0
            else:
                patience_ct += 1
                if patience_ct >= PATIENCE:
                    print("Arrêt anticipé : la perte de validation ne s'améliore plus.")
                    break

        if best_state is not None:
            model.load_state_dict(best_state)
        mlflow.log_metric("best_val_loss", best_val)

        poids = {k: v.contiguous() for k, v in model.state_dict().items()}
        save_file(poids, SORTIE_POIDS)
        mlflow.log_artifact(SORTIE_POIDS)
        print(f"Meilleurs poids sauvegardés dans {SORTIE_POIDS}")


if __name__ == "__main__":
    entrainer()

Overwriting train.py


In [10]:
import os
print("Fichiers en cache :", len(os.listdir("features_demo")) if os.path.exists("features_demo") else "dossier absent")

Fichiers en cache : 20000


In [11]:
import os
from kaggle_secrets import UserSecretsClient

# Je récupère l'adresse de mon serveur MLflow depuis le coffre-fort Kaggle,
# puis je la place dans une variable d'environnement, là où mon script la lit.
user_secrets = UserSecretsClient()
os.environ["MLFLOW_TRACKING_URI"] = user_secrets.get_secret("MLFLOW_TRACKING_URI")

print("Adresse MLflow chargée :", os.environ["MLFLOW_TRACKING_URI"])

Adresse MLflow chargée : https://auditeur-mlflow-hbb4ambmcnhhfjcz.swedencentral-01.azurewebsites.net


In [12]:
import train

# Pour ce premier essai, je limite volontairement le nombre de paires afin de
# valider toute la chaîne rapidement. Je repasserai ensuite à None (tout le jeu).
train.N_SAMPLES = None

train.entrainer()

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Périphérique : cuda


config.json:   0%|          | 0.00/803 [00:00<?, ?B/s]

configuration_cxrbert.py:   0%|          | 0.00/888 [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/BiomedVLP-BioViL-T:
- configuration_cxrbert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

CXRBertModel LOAD REPORT from: microsoft/BiomedVLP-BioViL-T
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.pooler.dense.bias       | UNEXPECTED |  | 
bert.pooler.dense.weight     | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 110M/110M [00:02<00:00, 39.4MB/s] 


20000 paires sélectionnées


Extraction des caractéristiques: 100%|██████████| 20000/20000 [00:00<00:00, 132044.18it/s]

0 nouvelles paires extraites ; les autres étaient déjà en cache.


Serveur MLflow : https://auditeur-mlflow-hbb4ambmcnhhfjcz.swedencentral-01.azurewebsites.net
Epoch 1/100  train_loss=0.6004  val_loss=0.5477  val_acc=0.7272
Epoch 2/100  train_loss=0.5145  val_loss=0.4935  val_acc=0.7752
Epoch 3/100  train_loss=0.4404  val_loss=0.4232  val_acc=0.8087
Epoch 4/100  train_loss=0.3798  val_loss=0.4144  val_acc=0.8165
Epoch 5/100  train_loss=0.3422  val_loss=0.3776  val_acc=0.8375
Epoch 6/100  train_loss=0.3215  val_loss=0.3705  val_acc=0.8390
Epoch 7/100  train_loss=0.3022  val_loss=0.3537  val_acc=0.8468
Epoch 8/100  train_loss=0.2872  val_loss=0.3278  val_acc=0.8652
Epoch 9/100  train_loss=0.2718  val_loss=0.3693  val_acc=0.8462
Epoch 10/100  train_loss=0.2633  val_loss=0.3124  val_acc=0.8718
Epoch 11/100  train_loss=0.2537  val_loss=0.3048  val_acc=0.8790
Epoch 12/100  train_loss=0.2459  val_loss=0.3436  val_acc=0.8602
Epoch 13/100  train_loss=0.2334  val_loss=0.3182  val_acc=0.8738
Epoch 14/100  train_loss=0.2264  val_loss=0.2987  val_acc=0.8798
Epoch 

In [13]:
import os

base = "/kaggle/input/datasets/aichajedha/auditeur-chexpert-20000"
for racine, dossiers, fichiers in os.walk(base):
    niveau = racine.replace(base, "").count(os.sep)
    if niveau <= 3:
        print(racine, "->", dossiers[:5], f"({len(fichiers)} fichiers)")

/kaggle/input/datasets/aichajedha/auditeur-chexpert-20000 -> ['images'] (1 fichiers)
/kaggle/input/datasets/aichajedha/auditeur-chexpert-20000/images -> ['images'] (0 fichiers)
/kaggle/input/datasets/aichajedha/auditeur-chexpert-20000/images/images -> ['valid', 'train'] (0 fichiers)
/kaggle/input/datasets/aichajedha/auditeur-chexpert-20000/images/images/valid -> ['patient64611', 'patient64545', 'patient64711', 'patient64684', 'patient64648'] (0 fichiers)
/kaggle/input/datasets/aichajedha/auditeur-chexpert-20000/images/images/train -> ['patient07547', 'patient40657', 'patient23363', 'patient24490', 'patient58373'] (0 fichiers)


In [14]:
import os
import pandas as pd

BASE = "/kaggle/input/datasets/aichajedha/auditeur-chexpert-20000"
IMG_BASE = os.path.join(BASE, "images", "images")
CSV = os.path.join(BASE, "chexpert_sample_dataset.csv")

df = pd.read_csv(CSV)
print("Lignes dans le CSV :", len(df))
print("Colonnes :", list(df.columns))

# On teste l'existence des images pour les 5 premières lignes
for rel in df["path_to_image"].head(5):
    chemin = os.path.join(IMG_BASE, rel)
    print(os.path.exists(chemin), "->", chemin)

Lignes dans le CSV : 20000
Colonnes : ['Unnamed: 0', 'path_to_image', 'deid_patient_id', 'report', 'sub_report', 'target', 'report_length', 'sub_report_length']
True -> /kaggle/input/datasets/aichajedha/auditeur-chexpert-20000/images/images/train/patient53048/study1/view1_frontal.jpg
True -> /kaggle/input/datasets/aichajedha/auditeur-chexpert-20000/images/images/train/patient52893/study1/view1_frontal.jpg
True -> /kaggle/input/datasets/aichajedha/auditeur-chexpert-20000/images/images/train/patient14924/study1/view1_frontal.jpg
True -> /kaggle/input/datasets/aichajedha/auditeur-chexpert-20000/images/images/train/patient40061/study1/view1_frontal.jpg
True -> /kaggle/input/datasets/aichajedha/auditeur-chexpert-20000/images/images/train/patient49019/study1/view1_frontal.jpg
